In [1]:
import numpy as np
import pandas as pd
import warnings
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import torch

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

%env JOBLIB_TEMP_FOLDER=/tmp

True
NVIDIA GeForce RTX 3060
env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
folder_path = "dataset/"
train = pd.read_pickle(f"{folder_path}merged_train.pkl")
test = pd.read_pickle(f"{folder_path}merged_test.pkl")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

train = train.sort_values("TransactionDT")

# Drop target and TransactionID
X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]

# Time-based 80-20 split
split_idx = int(len(train) * 0.8)

X_train = X.iloc[:split_idx]
X_valid = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

In [4]:
# Random Forest
# Train model
rf_model = RandomForestClassifier()

rf_model.fit(X_train, y_train)

# Evaluate
y_pred_prob = rf_model.predict_proba(X_valid)[:, 1]
rf_roc_auc = roc_auc_score(y_valid, y_pred_prob)

In [5]:
# LighGBM
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
)

lgb_model.fit(X_train, y_train)

# Fraud probabilities
y_pred_prob = lgb_model.predict_proba(X_valid)[:, 1]

# ROC-AUC score
lgb_roc_auc = roc_auc_score(y_valid, y_pred_prob)

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.413141 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35129
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 438
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
)

xgb_model.fit(X_train, y_train)

# Fraud probabilities
y_pred_prob = xgb_model.predict_proba(X_valid)[:, 1]

# ROC-AUC score
xgb_roc_auc = roc_auc_score(y_valid, y_pred_prob)

In [8]:
print("Random Forest")
print(f"ROC-AUC: {rf_roc_auc:.4f}")
print("LightGBM")
print(f"ROC-AUC: {lgb_roc_auc:.4f}")
print("XGBoost")
print(f"ROC-AUC: {xgb_roc_auc:.4f}")

Random Forest
ROC-AUC: 0.8881
LightGBM
ROC-AUC: 0.9082
XGBoost
ROC-AUC: 0.9093


In [9]:
lgb_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": lgb_model.feature_importances_,
    }
)

lgb_importance_df.sort_values(by="Importance", ascending=False, inplace=True)
lgb_importance_df.head(20)

,Feature,Importance
0,TransactionDT,579
4,card2,572
9,addr1,544
3,card1,535
1,TransactionAmt,499
437,uid,498
27,C13,334
43,D15,298
13,P_emaildomain,291
438,uid2,284


In [10]:
xgb_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": xgb_model.feature_importances_,
    }
)

xgb_importance_df.sort_values(by="Importance", ascending=False, inplace=True)
xgb_importance_df.head(20)

,Feature,Importance
310,V258,0.220747396349907
21,C7,0.030132768675685
346,V294,0.026592621579766
201,V149,0.023558843880892
122,V70,0.023319888859987
143,V91,0.021896729245782
385,V333,0.017862567678094
18,C4,0.017383627593517
194,V142,0.015390029177070
22,C8,0.014951927587390


In [11]:
def top_features_by_prefix(df, prefix, n=20):
    return (
        df[df["Feature"].str.startswith(prefix)]
        .sort_values(by="Importance", ascending=False)
        .head(n)
    )

In [12]:
# Usage
print("LightGBM")
print(top_features_by_prefix(lgb_importance_df, "C"))
print(top_features_by_prefix(lgb_importance_df, "D"))
print(top_features_by_prefix(lgb_importance_df, "M"))
print(top_features_by_prefix(lgb_importance_df, "id_"))
print(top_features_by_prefix(lgb_importance_df, "V"))

LightGBM
   Feature  Importance
27     C13         334
15      C1         221
16      C2         187
28     C14         178
25     C11         162
20      C6         128
23      C9         118
19      C5          84
22      C8          80
26     C12          63
24     C10          45
17      C3          34
18      C4          30
21      C7          16
        Feature  Importance
43          D15         298
29           D1         246
434      DT_day         217
30           D2         209
38          D10         191
431  DeviceInfo         186
36           D8         172
32           D4         156
436     DT_hour         143
39          D11         112
42          D14          71
435  DT_weekday          66
41          D13          65
31           D3          65
40          D12          62
34           D6          53
33           D5          42
430  DeviceType          32
37           D9          28
433     DT_week          18
   Feature  Importance
47      M4          97
49      M6  

In [13]:
print("XGBoost")
print(top_features_by_prefix(xgb_importance_df, "C"))
print(top_features_by_prefix(xgb_importance_df, "D"))
print(top_features_by_prefix(xgb_importance_df, "M"))
print(top_features_by_prefix(xgb_importance_df, "id_"))
print(top_features_by_prefix(xgb_importance_df, "V"))

XGBoost
   Feature         Importance
21      C7  0.030132768675685
18      C4  0.017383627593517
22      C8  0.014951927587390
28     C14  0.014678808860481
15      C1  0.009497627615929
19      C5  0.004990960936993
27     C13  0.003931484185159
25     C11  0.003263124730438
20      C6  0.001699191168882
26     C12  0.001697053783573
16      C2  0.001552379224449
23      C9  0.001486074179411
24     C10  0.001370879821479
17      C3  0.000791784841567
        Feature         Importance
430  DeviceType  0.005029402673244
30           D2  0.003161146771163
31           D3  0.002988283056766
43          D15  0.001787903718650
29           D1  0.001449499628507
432    DT_month  0.001409214921296
41          D13  0.001390047720633
32           D4  0.001373261911795
433     DT_week  0.001361508737318
38          D10  0.001249377615750
33           D5  0.001230129506439
36           D8  0.001190923620015
39          D11  0.001088753924705
34           D6  0.001038121292368
431  DeviceInfo  

In [ ]:
import joblib
import pandas as pd

# Save datasets
SAVED_PATH = "saved/"
train.to_pickle(f"{SAVED_PATH}train_preprocessed.pkl")

X_train.to_pickle(f"{SAVED_PATH}X_train.pkl")
X_valid.to_pickle(f"{SAVED_PATH}X_valid.pkl")

y_train.to_pickle(f"{SAVED_PATH}y_train.pkl")
y_valid.to_pickle(f"{SAVED_PATH}y_valid.pkl")

# Save models
joblib.dump(rf_model, f"{SAVED_PATH}random_forest.pkl")
joblib.dump(lgb_model, f"{SAVED_PATH}lightgbm.pkl")
joblib.dump(xgb_model, f"{SAVED_PATH}xgboost.pkl")

# Save ROC-AUC results
results_df = pd.DataFrame(
    {
        "Model": [
            "Random Forest",
            "LightGBM",
            "XGBoost",
        ],
        "ROC-AUC": [
            rf_roc_auc,
            lgb_roc_auc,
            xgb_roc_auc,
        ],
    }
)

results_df.to_pickle(f"{SAVED_PATH}model_results.pkl")

# Save feature importances
# LightGBM
lgb_importance_df.sort_values(by="Importance", ascending=False, inplace=True)
lgb_importance_df.to_pickle(f"{SAVED_PATH}lightgbm_feature_importance.pkl")

# XGBoost
xgb_importance_df.sort_values(by="Importance", ascending=False, inplace=True)
xgb_importance_df.to_pickle(f"{SAVED_PATH}xgboost_feature_importance.pkl")

print("All files saved successfully.")

TypeError: NDFrame.to_pickle() got an unexpected keyword argument 'index'